### Transform Addresses data
#### 1. Create on record for each customer with 2 sets of address columns, 1 for shipping and 1 for billing address
#### 2. Write transformed data to the silver schema

In [0]:

df = spark.read.table('gizmobox_catalog_noori.bronze.py1_addresses')
display(df)

In [0]:
df.select('address_line_1', 
          'city', 
          'state', 
          'postcode',
          'address_type',
          'customer_id'
          ).display()

In [0]:
from pyspark.sql.functions import max

df_formatted = (df
                .groupBy('customer_id')
                .pivot('address_type',['shipping','billing'])
                .agg(
                    max("address_line_1").alias('address_line_1'),
                    max("city").alias('city'),
                    max("state").alias('state'),
                    max("postcode").alias('postcode')
                )
    )
display(df_formatted)



In [0]:
df_formatted.writeTo('gizmobox_catalog_noori.silver.py1_addresses').createOrReplace()

In [0]:
%sql
create table if not exists gizmobox_catalog_noori.silver.addresses
as
select * from (
select 
customer_id,
address_type,
address_line_1,
city,
state,
postcode 

from gizmobox_catalog_noori.bronze.v_addresses)
pivot(max(address_line_1) as address_line_1,
        max(city) as city,
        max(state) as state,
        max(postcode) as postalcode
for address_type
 in ('shipping','billing'));

In [0]:
spark.read.table('gizmobox_catalog_noori.silver.py1_addresses').display()